# RFR: проверка float32 на GPU в Colab

1. Откройте этот ноутбук в Google Colab и выберите **Runtime → Change runtime type → GPU**.
2. Запустите ячейки и загрузите подготовленный файл `colab_inputs.npz` из
   `downloads/numerai/float32_benchmark/`.

В VS Code выберите ядро **Colab** и среду с GPU. Перед запуском загрузите
`colab_inputs.npz` через контекстное меню файла **Upload to Colab**.
Ноутбук сначала ищет файл в `/content/colab_inputs.npz`, затем в текущей папке.
Сохранённые CSV можно получить из среды Colab; браузерная загрузка результатов
в последней ячейке отключена по умолчанию для совместимости с VS Code.

Файл содержит ту же обучающую выборку, 256 опорных точек и первые 64 полные эры
валидации. Скачивать полные данные Numerai заново не требуется.
Это предварительная проверка скорости и точности, а не оценка на всех 640 эрах.
Опорные точки фиксированы; формула Kf сохранена. Постоянная часть C*n перенесена
в свободный член, чтобы уменьшить потерю точности float32.
Этот GPU-ноутбук ещё не запускался: локального NVIDIA GPU нет.


In [ ]:
%pip -q install numerai-tools==0.4.3 scipy pandas threadpoolctl

In [ ]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from scipy.spatial.distance import cdist
from numerai_tools.scoring import numerai_corr
from threadpoolctl import threadpool_limits
from google.colab import files

assert torch.cuda.is_available(), 'Выберите GPU в настройках среды выполнения Colab'
torch.set_num_threads(2)
cpu_limits = threadpool_limits(limits=2)
torch.backends.cuda.matmul.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
print(torch.__version__, torch.cuda.get_device_name(0))
data_path = Path('/content/colab_inputs.npz')
if not data_path.exists():
    data_path = Path('colab_inputs.npz')
if not data_path.exists():
    uploaded = files.upload()  # browser Colab; in VS Code use Upload to Colab first
    assert 'colab_inputs.npz' in uploaded, 'Загрузите colab_inputs.npz'
    del uploaded
    data_path = Path('colab_inputs.npz')
data = np.load(data_path, allow_pickle=False)
C, n, t, alpha = [float(data[key]) for key in ('C', 'n', 't', 'alpha')]
assert t == C*n, 'Эта ускоренная реализация проверена для непрерывной Kf с t=C*n'
X = data['X_validation']
y = data['y_validation']
eras = data['eras']
reference = data['reference']
points = data['supports']
q = data['coefficients']
bias = float(data['intercept']) + C*n*q.sum(dtype=np.float64)
print('Train:', data['X_train'].shape, 'Validation:', X.shape, 'Supports:', points.shape)


In [ ]:
def kernel_torch(x, points):
    # The original Kf minus the constant C*n; all tensors are float32.
    d2 = (x*x).sum(1)[:, None] + (points*points).sum(1)[None, :] - 2*(x @ points.T)
    d2 = d2.clamp_min(0)
    safe = torch.where(d2 > 0, d2, torch.ones_like(d2))
    return C*d2*torch.log(safe/t)

@torch.inference_mode()
def predict_tensor(x, points, coefficients, bias, batch_size=8192):
    out = torch.empty(len(x), dtype=x.dtype, device=x.device)
    for start in range(0, len(x), batch_size):
        stop = min(start+batch_size, len(x))
        out[start:stop] = kernel_torch(x[start:stop], points) @ coefficients + bias
    return out

def cpu_original():
    out = np.empty(len(X), dtype=np.float64)
    for start in range(0, len(X), 1024):
        stop = min(start+1024, len(X))
        d2 = cdist(X[start:stop], points, metric='sqeuclidean')
        safe = np.where(d2 > 0, d2, 1)
        K = np.where(d2 == 0, t, C*(d2*np.log(safe/t)+n))
        out[start:stop] = K @ q + float(data['intercept'])
    return out

def measure(function, repeats=3):
    function()  # warmup
    torch.cuda.synchronize()
    durations = []
    for _ in range(repeats):
        torch.cuda.synchronize()
        tick = time.perf_counter()
        value = function()
        torch.cuda.synchronize()
        durations.append(time.perf_counter()-tick)
    return value, float(np.median(durations))


In [ ]:
cpu_x = torch.from_numpy(X)
cpu_points = torch.tensor(points, dtype=torch.float32)
cpu_q = torch.tensor(q, dtype=torch.float32)
gpu_x = cpu_x.cuda()
gpu_points = cpu_points.cuda()
gpu_q = cpu_q.cuda()
predictions, timings = {}, {}
predictions['CPU original float64'], timings['CPU original float64'] = measure(cpu_original)
np.testing.assert_allclose(predictions['CPU original float64'], reference, rtol=1e-7, atol=1e-8)
result, timings['PyTorch CPU float32'] = measure(lambda: predict_tensor(cpu_x, cpu_points, cpu_q, bias))
predictions['PyTorch CPU float32'] = result.numpy()
result, timings['GPU float32 compute'] = measure(lambda: predict_tensor(gpu_x, gpu_points, gpu_q, bias))
predictions['GPU float32 compute'] = result.cpu().numpy()

def end_to_end():
    return predict_tensor(cpu_x.cuda(), gpu_points, gpu_q, bias).cpu().numpy()
predictions['GPU float32 with transfer'], timings['GPU float32 with transfer'] = measure(end_to_end)
pd.DataFrame({'seconds': timings, 'speedup_vs_CPU': {key: timings['CPU original float64']/value
                                                   for key, value in timings.items()}})


## Обучение в float32 на тех же опорных точках

Решается та же Ridge-регрессия с alpha=0.1 и свободным членом через SVD.
Время включает передачу train на GPU и построение матрицы Kf. Предыдущая ячейка
проверяет только инференс с фиксированными коэффициентами; следующая отдельно
проверяет влияние float32 на обучение.


In [ ]:
@torch.inference_mode()
def fit_gpu():
    torch.cuda.synchronize()
    tick = time.perf_counter()
    train_x = torch.tensor(data['X_train'], dtype=torch.float32, device='cuda')
    train_y = torch.tensor(data['y_train'], dtype=torch.float32, device='cuda')
    design = torch.empty((len(train_x), len(gpu_points)), device='cuda', dtype=torch.float32)
    for start in range(0, len(train_x), 8192):
        stop = min(start+8192, len(train_x))
        design[start:stop] = kernel_torch(train_x[start:stop], gpu_points)
    x_mean = design.mean(0)
    y_mean = train_y.mean()
    design -= x_mean
    rhs = train_y-y_mean
    U, S, Vh = torch.linalg.svd(design, full_matrices=False)
    coefficients = Vh.T @ ((S/(S*S+alpha))*(U.T @ rhs))
    intercept = y_mean-x_mean @ coefficients
    torch.cuda.synchronize()
    return coefficients, intercept, time.perf_counter()-tick

new_q, new_bias, fit_seconds = fit_gpu()
print('GPU float32 fit seconds:', fit_seconds)
result, timings['GPU float32 refit'] = measure(lambda: predict_tensor(gpu_x, gpu_points, new_q, new_bias))
predictions['GPU float32 refit'] = result.cpu().numpy()


In [ ]:
scores = []
for era in np.unique(eras):
    mask = eras == era
    era_predictions = pd.DataFrame({key: value[mask] for key, value in predictions.items()})
    corr = numerai_corr(era_predictions, pd.Series(y[mask], name='target'),
                        max_filtered_index_ratio=0.0)
    scores.append({'era': int(era), **corr.to_dict()})
scores = pd.DataFrame(scores).set_index('era')
report = pd.DataFrame({
    'mean_corr': scores.mean(),
    'corr_sharpe': scores.mean()/scores.std(ddof=0),
    'predict_seconds': timings,
    'rmse_vs_reference': {key: np.sqrt(np.mean((value.astype(np.float64)-reference)**2))
                           for key, value in predictions.items()},
    'max_abs_error': {key: np.max(np.abs(value.astype(np.float64)-reference))
                      for key, value in predictions.items()},
})
report['speedup_vs_CPU'] = timings['CPU original float64']/report.predict_seconds
display(report)
report.to_csv('gpu_float32_summary.csv')
scores.to_csv('gpu_float32_per_era_corr.csv')
DOWNLOAD_IN_BROWSER = False  # set True when using the Colab website
if DOWNLOAD_IN_BROWSER:
    files.download('gpu_float32_summary.csv')
